In [10]:
import os, json, time
from google import genai
from dotenv import load_dotenv
import mubit, mubit.learn

load_dotenv()

SESSION = f"learn-demo-{int(time.time())}"
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"
pp = lambda obj: print(json.dumps(obj, indent=2, default=str))

# --- THE ONE LINE ---
run = mubit.learn.init(
    api_key=os.environ["MUBIT_API_KEY"],
    endpoint=os.environ.get("MUBIT_ENDPOINT", "http://127.0.0.1:3000"),
    agent_id="demo-agent",
    session_id=SESSION,
    inject_lessons=True,
    auto_reflect=True,
    auto_extract=True,
    max_token_budget=800,
    cache_ttl_seconds=5,
)

llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print(f"Session: {SESSION}")
print(f"LLM: {GEMINI_MODEL}")
print("Ready — all Gemini calls are now auto-instrumented.")

Session: learn-demo-1773847848
LLM: gemini-3.1-flash-lite-preview
Ready — all Gemini calls are now auto-instrumented.


In [11]:
response_1 = llm.models.generate_content(
    model=GEMINI_MODEL,
    contents=(
        "You are a senior SRE investigating a production incident. "
        "A developer rotated the JWT signing key but forgot to flush the Redis token cache first. "
        "As a result, stale tokens signed with the old key were served from Redis for 5 minutes, "
        "causing widespread auth failures.\n\n"
        "Write a concise post-mortem summary. Include:\n"
        "- Root cause\n"
        "- What the fix was\n"
        "- Rules to prevent this in future (use 'always' and 'never' phrasing)\n"
        "Keep it under 200 words."
    ),
)
print(response_1.text)

### Post-Mortem: JWT Key Rotation Incident

**Root Cause:**
The rotation process failed because the system attempted to force a transition by clearing the Redis token cache. This caused an immediate invalidation of all active sessions. Furthermore, the new signing key was not properly propagated to the JWKS before signing began, leading to a TOCTOU (Time-of-Check to Time-of-Use) window where tokens were rejected by Resource Servers.

**Resolution:**
We restored service by reverting to the previous signing key to allow active sessions to complete their lifecycle. We then updated the JWKS to include both the old and new keys, ensuring the new key was fully distributed before initiating the signing transition.

**Preventative Rules:**
*   **Never** flush the Redis token cache during key rotation, as it causes unnecessary global logouts.
*   **Always** implement a gradual Key Rollover strategy by maintaining old public keys in the JWKS until all tokens signed by them have naturally expired

In [12]:
# Wait for auto-ingestion + embedding, then check what was captured
time.sleep(8)

query_client = mubit.Client(
    endpoint=os.environ.get("MUBIT_ENDPOINT", "http://127.0.0.1:3000"),
    api_key=os.environ["MUBIT_API_KEY"],
    run_id=SESSION,
)
query_client.set_transport("http")

health = query_client.memory_health(session_id=SESSION, limit=100)
print(f"Memory after call #1: {health.get('entry_counts', {})}")
print("\nAll auto-extracted — zero manual calls.")

Memory after call #1: {}

All auto-extracted — zero manual calls.


## Call #2 — Different question, lessons auto-injected

mubit.learn fetches the rules/lessons extracted from call #1 and injects them into this prompt automatically.

In [13]:
mubit.learn._lesson_cache.clear()

response_2 = llm.models.generate_content(
    model=GEMINI_MODEL,
    contents="I need to rotate our JWT signing keys this weekend. What steps should I follow to avoid any downtime?",
)
print(response_2.text)

To rotate your JWT signing keys without downtime, you must follow a phased strategy that prevents the "Time-of-Check to Time-of-Use" (TOCTOU) error, where a service receives a token it cannot verify.

Based on our active security rules, here is the step-by-step procedure:

### Phase 1: Pre-publication (The "Addition" Phase)
Before you generate any tokens with the new key, you must update your infrastructure.
1.  **Generate the New Key Pair:** Create the new private/public key pair.
2.  **Update the JWKS Endpoint:** Publish the **new public key** to your JSON Web Key Set (JWKS) endpoint. 
    *   **Crucial:** Do not remove the old key yet.
    *   **Adherence to rules:** By ensuring the new public key is available in the JWKS *before* you start signing with the new key, you eliminate the risk of Resource Servers rejecting tokens because they don't recognize the new `kid`.

### Phase 2: Rollover (The "Signing" Phase)
Once you have verified that the new public key is successfully cached o

In [14]:
# End run — triggers automatic reflection
run.end()
time.sleep(5)

health = query_client.memory_health(session_id=SESSION, limit=100)
print(f"Final memory: {health.get('entry_counts', {})}")

lessons = query_client.control.lessons({"run_id": SESSION, "limit": 10})
for l in lessons.get("lessons", []):
    print(f"  [{l.get('lesson_type', '?'):12s}] {l.get('content', '')[:120]}...")

Final memory: {}


---

**Total Mubit-specific code: `mubit.learn.init(...)` — one line.**

Everything else — extraction, injection, reflection — was automatic.